# Task 3: ulang V3 (cls) dengan profil Task 2 JobBERT (Colab GPU)

Satu run (~6 menit di L4): V3 cls lr 2e-5, hanya profil Field/Sub-field/Level yang diganti ke `profiles_jobbert.parquet` (model Task 2 JobBERT, Level dari Task 1, sama seperti demo). Hasil lama di Drive tidak ditimpa: tag baru `V3-jbprof_cls_lr2e-05`.

Jalankan semua cell berurutan.

In [1]:
BRANCH = "docs/readme-demo"
PROFILES = "profiles_jobbert.parquet"   # data/cache/: profil Task 2 (Field/Sub-field model Task 2, Level dari Task 1 seperti CLI)
COPY_MODELS = True
CKPT_ON_DRIVE = False

from google.colab import drive, userdata
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/task3_match"
CKPT = f"{OUT}/_ckpt" if CKPT_ON_DRIVE else "/content/ckpt"
!mkdir -p {OUT}/results {OUT}/models {OUT}/logs {CKPT}
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [2]:
import os, subprocess
tok = userdata.get("GITHUB_TOKEN")
REPO = "github.com/Ferdin-Arsenic/Job_Matching_System-NLP-.git"

def git(*args):
    """Jalankan git tanpa mencetak token: error disensor, dan remote disimpan tanpa token."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode: raise RuntimeError("git gagal: " + (r.stderr or r.stdout).replace(tok, "***"))
    return r.stdout

%cd /content
if not os.path.exists("/content/repo"):
    git("clone", "-q", "-b", BRANCH, f"https://x-access-token:{tok}@{REPO}", "repo")
    git("-C", "repo", "remote", "set-url", "origin", f"https://{REPO}")   # token tidak tersimpan di .git/config
else:
    git("-C", "repo", "pull", "-q", f"https://x-access-token:{tok}@{REPO}", BRANCH)
%cd /content/repo
print(git("log", "--oneline", "-1"))
!pip install -q datasets accelerate
!ls data/splits data/cache outputs/ner

/content
/content/repo
95ba265 feat : add ppt order

data/cache:
profiles_jobbert.metadata.json	profiles.parquet       sbert_minilm.npz
profiles_jobbert.parquet	profiles_stub.parquet  task2_metadata.json
profiles.metadata.json		profiles_t2.parquet    task2.parquet

data/splits:
task2.json  task3.json

outputs/ner:
ats_pairs.parquet    ner_ats_resume.parquet  t1_ner_results.csv
ner_atlas.parquet    README.md		     t1_seed_check.csv
ner_ats_job.parquet  t1_crf_check.csv	     t1_variant_comparison.csv


In [3]:
import glob, time, pandas as pd

def tag_of(variant, kind, lr):
    return f"{variant}{'-jbprof' if variant == 'V3' and 'jobbert' in PROFILES else ''}_{kind}_lr{lr:g}"

def run(variant, kind, lr):
    tag = tag_of(variant, kind, lr)
    if os.path.exists(f"{OUT}/results/task3_ce_{tag}.csv"):
        print("skip (sudah ada di Drive):", tag); return
    t0 = time.time()
    cmd = (f"set -o pipefail; python task3_match/cross_encoder.py --variant {variant} --run {kind} --lr {lr} "
           f"--profiles {PROFILES} --ckpt-dir {CKPT} 2>&1 | tee -a {OUT}/logs/{tag}.log")
    subprocess.run(["bash", "-c", cmd], check=True)
    os.system(f"cp data/results/task3_ce_{tag}.csv data/results/task3_pred_{tag}.parquet {OUT}/results/")
    if COPY_MODELS: os.system(f"cp -r models/match/{tag} {OUT}/models/ && rm -rf models/match/{tag}")
    print(f"selesai {tag} dalam {(time.time() - t0) / 60:.1f} menit")

def results():
    return pd.concat([pd.read_csv(f) for f in glob.glob(f"{OUT}/results/task3_ce_*.csv")], ignore_index=True)

In [4]:
run("V3", "cls", 2e-5)

selesai V3-jbprof_cls_lr2e-05 dalam 6.4 menit


In [5]:
r = pd.read_csv(f"{OUT}/results/task3_ce_V3-jbprof_cls_lr2e-05.csv"); print(r.to_string())

                   model variant  run       lr       best_ckpt                  profiles split            metric      value
0   Cross-Encoder V3 cls      V3  cls  0.00002  checkpoint-954  profiles_jobbert.parquet   val          macro_F1   0.453988
1   Cross-Encoder V3 cls      V3  cls  0.00002  checkpoint-954  profiles_jobbert.parquet   val               MAE  18.304744
2   Cross-Encoder V3 cls      V3  cls  0.00002  checkpoint-954  profiles_jobbert.parquet   val          Spearman   0.349932
3   Cross-Encoder V3 cls      V3  cls  0.00002  checkpoint-954  profiles_jobbert.parquet   val           ROC_AUC   0.699724
4   Cross-Encoder V3 cls      V3  cls  0.00002  checkpoint-954  profiles_jobbert.parquet   val          P_No Fit   0.612069
5   Cross-Encoder V3 cls      V3  cls  0.00002  checkpoint-954  profiles_jobbert.parquet   val          R_No Fit   0.665625
6   Cross-Encoder V3 cls      V3  cls  0.00002  checkpoint-954  profiles_jobbert.parquet   val         F1_No Fit   0.637725
7   Cros